# ComfortPy Advanced Domain Evaluation Tutorial

**Requires optional extras — install with `pip install comfortpy[full]`**

This notebook demonstrates all advanced physics-based domain modules:

| Section | Module | Extra Required | Library |
|---------|--------|---------------|---------|
| 1 | Psychrometrics | `[psychrometrics]` | PsychroLib |
| 2 | Ventilation (CO₂ decay) | `[psychrometrics]` | PsychroLib |
| 3 | Reverberation (RT60) | `[acoustics]` | python-acoustics |
| 4 | Speech Intelligibility (STI) | `[acoustics]` | pyroomacoustics |
| 5 | Color Quality (CRI/CCT) | `[color]` | colour-science |
| 6 | Daylighting | `[daylighting]` | pyradiance |
| 7 | Global IEQ Blending | — | blends advanced + simple |

**Graceful degradation**: If an extra is not installed, the cell will print a helpful `pip install` message instead of crashing.

In [1]:
import numpy as np
import traceback

def try_run(label, fn):
    """Run a function, catching ImportError for missing extras."""
    try:
        result = fn()
        print(f"  [OK] {label}")
        return result
    except ImportError as e:
        print(f"  [SKIP] {label}")
        print(f"        {e}")
        return None
    except Exception:
        print(f"  [ERROR] {label}")
        traceback.print_exc()
        return None

## 1. Psychrometrics (PsychroLib)

`get_psychrometrics()` returns full psychrometric properties of moist air: wet-bulb, dew point, enthalpy, humidity ratio, vapor pressure, specific volume, and degree of saturation.

In [2]:
from comfortpy import get_psychrometrics

def demo_psychrometrics():
    # Standard indoor conditions: 25C, 50% RH, sea level
    standard = get_psychrometrics(tdb=25.0, rh=0.50, pressure=101325.0)
    print(f"Standard (25C, 50% RH):")
    print(f"  Wet bulb:       {standard.twb:.2f} C")
    print(f"  Dew point:      {standard.tdew:.2f} C")
    print(f"  Enthalpy:       {standard.enthalpy:.0f} J/kg")
    print(f"  Humidity ratio: {standard.hum_ratio:.5f} kg/kg")
    print(f"  Vapor pressure: {standard.vapor_pressure:.1f} Pa")
    print(f"  Specific vol:   {standard.moist_air_volume:.4f} m3/kg")

    # Hot & humid
    hot = get_psychrometrics(tdb=35.0, rh=0.80, pressure=101325.0)
    print(f"\nHot & humid (35C, 80% RH):")
    print(f"  Wet bulb:   {hot.twb:.2f} C")
    print(f"  Dew point:  {hot.tdew:.2f} C")
    print(f"  Enthalpy:   {hot.enthalpy:.0f} J/kg")

    # High altitude (Denver ~1600m)
    denver = get_psychrometrics(tdb=20.0, rh=0.40, pressure=83500.0)
    print(f"\nDenver altitude (20C, 40% RH, 83.5 kPa):")
    print(f"  Wet bulb:     {denver.twb:.2f} C")
    print(f"  Specific vol: {denver.moist_air_volume:.4f} m3/kg")
    print(f"  (higher specific volume at altitude)")
    return standard

psych_result = try_run("get_psychrometrics", demo_psychrometrics)

Standard (25C, 50% RH):
  Wet bulb:       17.89 C
  Dew point:      13.86 C
  Enthalpy:       50322 J/kg
  Humidity ratio: 0.00988 kg/kg
  Vapor pressure: 1584.6 Pa
  Specific vol:   0.8580 m3/kg



Hot & humid (35C, 80% RH):
  Wet bulb:   31.81 C
  Dew point:  31.02 C
  Enthalpy:   109423 J/kg



Denver altitude (20C, 40% RH, 83.5 kPa):
  Wet bulb:     11.75 C
  Specific vol: 1.0192 m3/kg
  (higher specific volume at altitude)
  [OK] get_psychrometrics


## 2. Ventilation Rate from CO₂ Decay

`evaluate_ventilation()` estimates Air Change Rate (ACH) using either:
- **CO₂ decay method**: Fits an exponential to the decay phase when occupants leave
- **Steady-state method**: Uses room volume, occupant count, and CO₂ generation rate

In [3]:
from comfortpy import evaluate_ventilation

def demo_ventilation():
    # Simulate CO2 decay: 1500 ppm -> 420 ppm over 2 hours, ACH=5.0
    timestamps = np.arange(0, 7200, 300, dtype=float)  # 5-min intervals
    k = 5.0 / 3600.0
    co2 = 420.0 + (1500.0 - 420.0) * np.exp(-k * timestamps)

    result = evaluate_ventilation(
        co2=co2, timestamps=timestamps,
        outdoor_co2=420.0, occupancy_type="office",
    )
    print("CO2 decay method:")
    print(f"  ACH:                    {result.ach:.2f} /hr")
    print(f"  Method:                 {result.ach_method}")
    print(f"  Ventilation efficiency: {result.ventilation_efficiency:.1%}")
    print(f"  CO2 peak:               {result.co2_peak:.0f} ppm")
    print(f"  Compliant (>=2.0 ACH):  {bool(result.compliant)}")
    print(f"  Score:                  {result.score:.1f}/100")

    # Steady-state method
    co2_steady = np.full(20, 850.0)
    result_ss = evaluate_ventilation(
        co2=co2_steady, outdoor_co2=420.0,
        occupancy_type="classroom",
        room_volume=60.0, n_occupants=25,
    )
    print(f"\nSteady-state method (classroom, 25 occupants):")
    print(f"  ACH:                    {result_ss.ach:.2f} /hr")
    print(f"  Method:                 {result_ss.ach_method}")
    print(f"  Compliant (>=3.0 ACH):  {bool(result_ss.compliant)}")
    return result

vent_result = try_run("evaluate_ventilation", demo_ventilation)

CO2 decay method:
  ACH:                    4.04 /hr
  Method:                 co2_decay
  Ventilation efficiency: 100.0%
  CO2 peak:               1500 ppm
  Compliant (>=2.0 ACH):  True
  Score:                  100.0/100

Steady-state method (classroom, 25 occupants):
  ACH:                    17.44 /hr
  Method:                 steady_state
  Compliant (>=3.0 ACH):  True
  [OK] evaluate_ventilation


## 3. Reverberation Time (python-acoustics)

`evaluate_reverberation()` calculates RT60 using Sabine or Eyring formulas from room surface areas and absorption coefficients.

In [4]:
from comfortpy import evaluate_reverberation

def demo_reverberation():
    # 5x5x3 m office room = 75 m3
    # 5 surfaces, 6 octave bands (125-4000 Hz)
    surfaces = np.array([
        [25, 25, 25, 25, 25, 25],  # floor (5x5)
        [25, 25, 25, 25, 25, 25],  # ceiling (5x5)
        [15, 15, 15, 15, 15, 15],  # wall 1 (5x3)
        [15, 15, 15, 15, 15, 15],  # wall 2
        [15, 15, 15, 15, 15, 15],  # wall 3
    ])
    absorption = np.array([
        [0.05, 0.10, 0.15, 0.20, 0.25, 0.30],  # carpet
        [0.20, 0.55, 0.80, 0.85, 0.85, 0.85],  # acoustic ceiling
        [0.05, 0.05, 0.08, 0.10, 0.12, 0.15],  # painted wall
        [0.05, 0.05, 0.08, 0.10, 0.12, 0.15],  # painted wall
        [0.05, 0.05, 0.08, 0.10, 0.12, 0.15],  # painted wall
    ])

    sabine = evaluate_reverberation(
        surfaces=surfaces, absorption_coeffs=absorption,
        volume=75.0, method="sabine", room_type="office",
    )
    print(f"Sabine method (office, 75 m3):")
    print(f"  RT60:       {np.round(sabine.rt60, 3)} s")
    print(f"  Mean RT60:  {np.mean(sabine.rt60):.3f} s")
    print(f"  NRC:        {sabine.nrc:.2f}")
    print(f"  Compliant:  {bool(sabine.compliant)}")
    print(f"  Score:      {sabine.score:.1f}/100")

    eyring = evaluate_reverberation(
        surfaces=surfaces, absorption_coeffs=absorption,
        volume=75.0, method="eyring", room_type="office",
    )
    print(f"\nEyring method:")
    print(f"  Mean RT60:  {np.mean(eyring.rt60):.3f} s")
    print(f"  Score:      {eyring.score:.1f}/100")
    return sabine

reverb_result = try_run("evaluate_reverberation", demo_reverberation)

  [SKIP] evaluate_reverberation
        python-acoustics is required for reverberation calculations. Install it with: pip install comfortpy[acoustics]


## 4. Speech Intelligibility (pyroomacoustics)

`evaluate_speech_intelligibility()` estimates the Speech Transmission Index (STI) from a room impulse response.

In [5]:
from comfortpy import evaluate_speech_intelligibility

def demo_speech_intelligibility():
    # Synthetic impulse response: direct sound + exponential decay
    sample_rate = 16000
    duration = 0.5
    n_samples = int(sample_rate * duration)
    t = np.arange(n_samples) / sample_rate

    rt60_true = 0.3
    ir = np.exp(-3 * np.log(10) * t / rt60_true)
    ir[0] = 1.0
    ir += 0.01 * np.random.default_rng(42).standard_normal(n_samples)

    result = evaluate_speech_intelligibility(
        impulse_response=ir, sample_rate=sample_rate,
    )
    print(f"Synthetic IR (RT60 ~0.3s, 16 kHz):")
    print(f"  STI:               {result.sti:.3f}")
    print(f"  Rating:            {result.rating}")
    print(f"  Measured RT60:     {result.rt60_measured:.3f} s")
    print(f"  Compliant (>=0.60): {bool(result.compliant)}")
    print(f"  Score:             {result.score:.1f}/100")
    return result

sti_result = try_run("evaluate_speech_intelligibility", demo_speech_intelligibility)

  [SKIP] evaluate_speech_intelligibility
        pyroomacoustics is required for speech intelligibility calculations. Install it with: pip install comfortpy[acoustics]


## 5. Color Quality (colour-science)

`evaluate_color_quality()` calculates CRI (Ra), CCT, and D_uv from a spectral power distribution.

In [6]:
from comfortpy import evaluate_color_quality

def demo_color_quality():
    wavelengths = np.arange(380, 781, 1)  # 380-780 nm

    # Simulate a warm white LED SPD (Gaussian mixture)
    spd = (
        0.3 * np.exp(-0.5 * ((wavelengths - 450) / 15) ** 2)  # blue peak
        + 0.8 * np.exp(-0.5 * ((wavelengths - 560) / 30) ** 2)  # green peak
        + 1.0 * np.exp(-0.5 * ((wavelengths - 600) / 40) ** 2)  # red peak
    )

    result = evaluate_color_quality(
        spectral_distribution=spd, wavelengths=wavelengths,
        method="CIE 1995", min_cri=80.0, additional_data=True,
    )
    print(f"Warm white LED SPD:")
    print(f"  CRI (Ra):   {result.cri:.1f}")
    print(f"  CCT:        {result.cct:.0f} K")
    print(f"  D_uv:       {result.duv:.4f}")
    print(f"  Compliant:  {bool(result.compliant)}")
    print(f"  Score:      {result.score:.1f}/100")
    print(f"  Method:     {result.method}")
    return result

color_result = try_run("evaluate_color_quality", demo_color_quality)

  [SKIP] evaluate_color_quality
        colour-science is required for color quality evaluation. Install it with: pip install comfortpy[color]
Note: colour-science requires Python >= 3.11.


## 6. Daylighting (pyradiance)

`evaluate_daylighting()` runs a Radiance ray-tracing simulation to compute point-in-time illuminance and optionally Daylight Glare Probability (DGP).

> **Note**: This requires a pre-built Radiance octree file (`.oct`). On Windows, WSL is recommended for Radiance binaries.

In [7]:
from comfortpy import evaluate_daylighting

def demo_daylighting():
    # NOTE: Requires a pre-built Radiance octree file (.oct)
    # This demonstrates the API — you need a real octree to get results
    sensor_points = np.array([
        [2.5, 2.5, 0.8, 0, 0, 1],   # center of room, facing up
        [0.5, 0.5, 0.8, 0, 0, 1],   # corner
        [4.5, 4.5, 0.8, 0, 0, 1],   # opposite corner
    ])
    result = evaluate_daylighting(
        octree_file="room.oct",
        sensor_points=sensor_points,
        sky_model="cie_overcast",
        task_type="office_writing",
    )
    print(f"Daylighting (cie_overcast sky):")
    print(f"  Illuminance: {result.illuminance} lux")
    print(f"  DGP:         {result.dgp}")
    print(f"  Compliant:   {result.compliant}")
    print(f"  Score:       {result.score}")
    print(f"  Target lux:  {result.target_lux}")
    return result

daylight_result = try_run("evaluate_daylighting", demo_daylighting)

  [SKIP] evaluate_daylighting
        pyradiance is required for daylighting simulation. Install it with: pip install comfortpy[daylighting]
Note: On Windows, WSL is recommended for Radiance binaries.


## 7. Global IEQ Index — Simple + Advanced Blended

Advanced results can be passed alongside simple results to `calculate_global_ieq()`. When both are provided, advanced scores blend with or override simple scores:

| Advanced Result | Blending Logic |
|----------------|----------------|
| `daylighting` | Overrides visual score |
| `color_quality` | 70% visual + 30% color_quality |
| `reverberation` | 60% acoustic + 40% reverberation |
| `speech_intelligibility` | 50% acoustic + 50% STI |
| `ventilation` | Overrides IAQ score |

In [8]:
from comfortpy import (
    evaluate_thermal, evaluate_visual, evaluate_acoustic, evaluate_iaq,
    calculate_global_ieq,
)

# Simple domain results (always available)
rng = np.random.default_rng(42)
thermal = evaluate_thermal(
    tdb=np.array([24.0, 25.0, 26.0]),
    tr=np.array([24.0, 25.0, 26.0]),
    vr=np.array([0.1, 0.1, 0.1]),
    rh=np.array([50.0, 50.0, 50.0]),
    met=1.2, clo=0.5, category="B",
)
visual = evaluate_visual(illuminance=np.array([450, 520, 380]), task_type="office_writing")
acoustic = evaluate_acoustic(laeq=np.array([42, 48, 55]), nc_level="NC-35")
iaq = evaluate_iaq(co2=np.array([800, 950, 1100]), threshold_level="good")

# Build kwargs from whatever advanced results succeeded
kwargs = {"thermal": thermal, "visual": visual, "acoustic": acoustic, "iaq": iaq}

advanced_results = {
    "ventilation": vent_result,
    "reverberation": reverb_result,
    "speech_intelligibility": sti_result,
    "color_quality": color_result,
    "daylighting": daylight_result,
}

for name, result in advanced_results.items():
    if result is not None:
        kwargs[name] = result
        print(f"  Including: {name}")

if not any(k in kwargs for k in advanced_results):
    print("  (No advanced results available — showing simple-only index)")

result = calculate_global_ieq(**kwargs)

print(f"\n  Global IEQ Index: {np.round(result.index, 1)}")
print(f"\n  Domain scores:")
for domain, score in result.domain_scores.items():
    print(f"    {domain:20s}: {np.mean(score):.1f}")
print(f"  Weights used: {result.weights_used}")
print(f"  Domains:      {result.domains}")

  Including: ventilation

  Global IEQ Index: [82.2 78.8 67.6]

  Domain scores:
    thermal             : 93.0
    visual              : 88.4
    acoustic            : 20.0
    iaq                 : 73.3
  Weights used: {'thermal': 0.4, 'visual': 0.2, 'acoustic': 0.15, 'iaq': 0.25}
  Domains:      ['thermal', 'visual', 'acoustic', 'iaq']


## 8. SensorData.available_advanced_domains()

`SensorData` can auto-detect which advanced evaluations are possible based on the columns present in your DataFrame.

In [9]:
import pandas as pd
from comfortpy import SensorData

# Column names must match DEFAULT_ALIASES in data_handler.py
df = pd.DataFrame({
    "temperature": [22.0, 23.0, 24.0],      # -> air_temp_c
    "rh": [50, 55, 60],                     # -> relative_humidity_pct
    "co2": [800, 900, 1000],               # -> co2_ppm
    "lux": [400, 500, 450],                # -> illuminance_lux
    "room_volume": [75.0, 75.0, 75.0],     # -> room_volume_m3
})

sensor = SensorData(df)
available = sensor.available_advanced_domains()

print(f"Sensor columns: {list(df.columns)}")
print(f"Detected advanced domains: {available}")
print()
print("Mapping:")
print("  'psychrometrics'  <- temperature + rh columns")
print("  'ventilation'     <- co2 column")
print("  'daylighting'     <- lux (illuminance) column")
print("  'reverberation'   <- room_volume column")
print("  'color_quality'   <- (needs spectral_power column — not present)")

Sensor columns: ['temperature', 'rh', 'co2', 'lux', 'room_volume']
Detected advanced domains: ['daylighting', 'reverberation', 'ventilation', 'psychrometrics']

Mapping:
  'psychrometrics'  <- temperature + rh columns
  'ventilation'     <- co2 column
  'daylighting'     <- lux (illuminance) column
  'reverberation'   <- room_volume column
  'color_quality'   <- (needs spectral_power column — not present)


## Summary

Install all advanced extras with:
```bash
pip install comfortpy[full]
```

Or individually:
```bash
pip install comfortpy[psychrometrics]   # PsychroLib
pip install comfortpy[acoustics]        # python-acoustics + pyroomacoustics
pip install comfortpy[color]            # colour-science (Python >= 3.11)
pip install comfortpy[daylighting]      # pyradiance (WSL recommended on Windows)
```

Full documentation: **GUIDE.md Section 13 — Advanced Domain Evaluation**

# This cell intentionally left blank — delete it.